In [3]:
import pandas as pd
import numpy as np

anos = [2021, 2022, 2023, 2024, 2025]
dfs_bpp = []

for ano in anos:
    caminho = f"../data/raw/BPP/dfp_cia_aberta_BPP_con_{ano}.csv"
    df = pd.read_csv(caminho, sep=";", encoding="latin1")
    df["ANO_ARQUIVO"] = ano
    dfs_bpp.append(df)
    print(f"{ano}: {df.shape[0]} linhas")

df_bpp = pd.concat(dfs_bpp, ignore_index=True)
print("Total:", df_bpp.shape)

2021: 104017 linhas
2022: 109084 linhas
2023: 110540 linhas
2024: 107116 linhas
2025: 101380 linhas
Total: (532137, 15)


In [4]:
df_bpp[df_bpp["DS_CONTA"].str.contains("Patrimônio Líquido", case=False, na=False)][["CD_CONTA", "DS_CONTA"]].drop_duplicates().sort_values("CD_CONTA")

,CD_CONTA,DS_CONTA
13796,2.01.05.02.07,Patrimônio líquido negativo de investida
56234,2.02.02.02.08,Investimento com patrimônio líquido negativo
384,2.03,Patrimônio Líquido Consolidado
56542,2.03.04.11,Outros componentes do patrimônio líquido
11274,2.03.06.01,Ajuste do Patrimônio Líquido - Variação Cambia...
18162,2.03.08.01,Patrimônio líquido dos não controladores
48592,2.03.08.02,Patrimônio líquido atribuível aos proprietário...
34,2.07,Patrimônio Líquido Consolidado
36,2.07.01,Patrimônio Líquido Atribuído ao Controlador
82,2.07.02,Patrimônio Líquido Atribuído aos Não Controlad...


In [15]:
df_bpp.columns.to_list()


['CNPJ_CIA',
 'DT_REFER',
 'VERSAO',
 'DENOM_CIA',
 'CD_CVM',
 'GRUPO_DFP',
 'MOEDA',
 'ESCALA_MOEDA',
 'ORDEM_EXERC',
 'DT_FIM_EXERC',
 'CD_CONTA',
 'DS_CONTA',
 'VL_CONTA',
 'ST_CONTA_FIXA',
 'ANO_ARQUIVO']

In [16]:
df_bpp[df_bpp["DS_CONTA"].str.contains("Patrimônio Líquido", case=False, na=False)][["CD_CONTA", "DS_CONTA"]].drop_duplicates().sort_values("CD_CONTA")

,CD_CONTA,DS_CONTA
13796,2.01.05.02.07,Patrimônio líquido negativo de investida
56234,2.02.02.02.08,Investimento com patrimônio líquido negativo
384,2.03,Patrimônio Líquido Consolidado
56542,2.03.04.11,Outros componentes do patrimônio líquido
11274,2.03.06.01,Ajuste do Patrimônio Líquido - Variação Cambia...
18162,2.03.08.01,Patrimônio líquido dos não controladores
48592,2.03.08.02,Patrimônio líquido atribuível aos proprietário...
34,2.07,Patrimônio Líquido Consolidado
36,2.07.01,Patrimônio Líquido Atribuído ao Controlador
82,2.07.02,Patrimônio Líquido Atribuído aos Não Controlad...


In [17]:
pl_codigos = ["2.03", "2.07", "2.08"]
df_bpp[df_bpp["CD_CONTA"].isin(pl_codigos)].groupby("CD_CONTA")["CNPJ_CIA"].nunique()

CD_CONTA
2.03    537
2.07     25
2.08     11
Name: CNPJ_CIA, dtype: int64

In [20]:
pl_codigos = ["2.03", "2.07", "2.08"]
df_pl = df_bpp[df_bpp["CD_CONTA"].isin(pl_codigos)].copy()

df_pl_ano = df_pl.groupby(["DENOM_CIA", "ANO_ARQUIVO", "CD_CONTA"])["VL_CONTA"].sum().reset_index()
df_pl_ano.head(20)

,DENOM_CIA,ANO_ARQUIVO,CD_CONTA,VL_CONTA
0,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2021,2.03,204641.0
1,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2022,2.03,162103.0
2,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2023,2.03,48370.0
3,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2024,2.03,-551784.0
4,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2021,2.03,8112366.0
5,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2022,2.03,12359054.0
6,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2023,2.03,16982939.0
7,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2024,2.03,22360143.0
8,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2025,2.03,11086217.0
9,AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. ...,2021,2.03,1964899.0


In [21]:
#Teste mais rigoroso em relação a ver se repete ou não
contagem = df_pl.groupby(["DENOM_CIA", "ANO_ARQUIVO"])["CD_CONTA"].nunique()
print(contagem[contagem > 1])  # se vier vazio, nenhuma empresa/ano tem mais de 1 código — confirma 100%

DENOM_CIA                                   ANO_ARQUIVO
BANCO BMG S/A                               2021           3
                                            2022           3
                                            2023           3
                                            2024           3
                                            2025           3
                                                          ..
SUL 116 PARTICIPAÇÕES S.A. - EM LIQUIDAÇÃO  2023           3
XP INVESTIMENTOS S.A.                       2021           2
                                            2022           2
                                            2023           2
                                            2024           2
Name: CD_CONTA, Length: 94, dtype: int64


In [32]:
df_pl[(df_pl["DENOM_CIA"] == "BANCO BMG S/A") | (df_pl["DENOM_CIA"] == "XP INVESTIMENTOS S.A.")][["DENOM_CIA","CD_CONTA", "DS_CONTA", "VL_CONTA","ANO_ARQUIVO"]].sort_values("ANO_ARQUIVO")

,DENOM_CIA,CD_CONTA,DS_CONTA,VL_CONTA,ANO_ARQUIVO
48632,XP INVESTIMENTOS S.A.,2.03,Provisões,19711.0,2021
48633,XP INVESTIMENTOS S.A.,2.03,Provisões,29308.0,2021
48660,XP INVESTIMENTOS S.A.,2.07,Patrimônio Líquido Consolidado,4043172.0,2021
48661,XP INVESTIMENTOS S.A.,2.07,Patrimônio Líquido Consolidado,6705725.0,2021
86167,BANCO BMG S/A,2.03,Passivos Financeiros ao Custo Amortizado,21527471.0,2021
86168,BANCO BMG S/A,2.03,Passivos Financeiros ao Custo Amortizado,28040991.0,2021
86193,BANCO BMG S/A,2.07,Passivos sobre Ativos Não Correntes a Venda e ...,0.0,2021
86194,BANCO BMG S/A,2.07,Passivos sobre Ativos Não Correntes a Venda e ...,0.0,2021
86199,BANCO BMG S/A,2.08,Patrimônio Líquido Consolidado,4152428.0,2021
86200,BANCO BMG S/A,2.08,Patrimônio Líquido Consolidado,3823900.0,2021


In [33]:

df_pl = df_bpp[df_bpp["DS_CONTA"] == "Patrimônio Líquido Consolidado"].copy()

In [42]:
df_pl = df_bpp[
    (df_bpp["DS_CONTA"] == "Patrimônio Líquido Consolidado") &
    (df_bpp["ORDEM_EXERC"] == "ÚLTIMO")
].copy()

df_pl_ano = df_pl.groupby(["DENOM_CIA", "ANO_ARQUIVO"])["VL_CONTA"].sum().reset_index()


In [43]:
df_pl_ano

,DENOM_CIA,ANO_ARQUIVO,VL_CONTA
0,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2021,44981.0
1,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2022,125786.0
2,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2023,-77416.0
3,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2024,-474368.0
4,AEGEA SANEAMENTO E PARTICIPAÇÕES S.A.,2021,6403746.0
...,...,...,...
2313,ZAMP SA,2021,1572720.0
2314,ZAMP SA,2022,1485188.0
2315,ZAMP SA,2023,1393680.0
2316,ZAMP SA,2024,1546026.0


In [44]:
contagem = df_pl.groupby(["DENOM_CIA", "ANO_ARQUIVO"])["CD_CONTA"].nunique()
print(contagem[contagem > 1])

Series([], Name: CD_CONTA, dtype: int64)


In [48]:
df_pl_ano.to_csv("../data/processed/patrimonio_liquido_2021_2025.csv", index=False)

In [49]:
df_bpp[
    (df_bpp["DENOM_CIA"] == "TIM S.A.") &
    (df_bpp["ANO_ARQUIVO"] == 2024) &
    (df_bpp["DS_CONTA"].str.contains("Patrim", case=False, na=False))
][["CD_CONTA", "DS_CONTA", "VL_CONTA", "ORDEM_EXERC"]]


,CD_CONTA,DS_CONTA,VL_CONTA,ORDEM_EXERC
333101,2.03,Patrimônio Líquido Consolidado,26015940.0,PENÚLTIMO
333102,2.03,Patrimônio Líquido Consolidado,0.0,ÚLTIMO
333145,2.03.06,Ajustes de Avaliação Patrimonial,0.0,PENÚLTIMO
333146,2.03.06,Ajustes de Avaliação Patrimonial,0.0,ÚLTIMO
